"LoRA v3 — Deep MLP + Attention Fine-tuning"
What's different from v2:

v2 adapted only attention layers (q, k, v, o projections) — this controls what the model looks at
v3 adds MLP layers (gate, up, down projections) — this controls how the model reasons about what it sees

In a transformer, attention layers decide which parts of the image and text to focus on, while MLP layers do the actual computation and reasoning. Your errors showed the model focuses on the right things but picks the wrong answer — that's a reasoning problem, which is exactly what MLP adaptation targets.
Summary: ~3.5M params (vs 1.1M in v2), 3 epochs (vs 2), higher dropout (0.1 vs 0.05) for regularization, same learning rate.

## Cell 0: Install

In [1]:
!pip install -q "transformers==4.57.1" "peft==0.18.0" bitsandbytes accelerate "pandas==2.2.2" "pillow==11.3.0" tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.3 MB/s eta 0:00:00


## Cell 1: Download Data

In [2]:
import kagglehub
from pathlib import Path
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
COMPETITION_NAME = "pixels-to-predictions"
COMP_DIR = Path(kagglehub.competition_download(COMPETITION_NAME))
print("Downloaded to:", COMP_DIR)

100%|██████████| 358M/358M [00:09<00:00, 38.8MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/competitions/pixels-to-predictions


## Cell 2: Imports & Config

In [9]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json, re, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForVision2Seq, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
IMG_SIZE = 224
CHOICE_LETTERS = "ABCDEFGHIJ"

# Training
BATCH_SIZE_TRAIN = 1
GRAD_ACCUM_STEPS = 16
LR = 2e-5
EPOCHS = 3
WARMUP_RATIO = 0.05

# LoRA (~4.78M params — MLP + attention)
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Inference
BATCH_SIZE_INFER = 4

SAVE_DIR = Path("/content/lora_v3")

from google.colab import drive
drive.mount("/content/drive")
DRIVE_SAVE = Path("/content/drive/MyDrive/lora_v3_best")

print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  {torch.cuda.get_device_name(0)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: True
  Tesla T4


## Cell 3: Load Data

In [10]:
DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists():
    DATA_DIR = COMP_DIR

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

example_name = Path(val_df.iloc[0]["image_path"]).name
matches = list(COMP_DIR.rglob(example_name))
DATA_ROOT = matches[0].parents[2]
print("DATA_ROOT:", DATA_ROOT)

Train: 3109 | Val: 1048 | Test: 1008
DATA_ROOT: /root/.cache/kagglehub/competitions/pixels-to-predictions/images


## Cell 4: Prompt + Image Helpers

In [11]:
def clean_text(x):
    if pd.isna(x): return ""
    return str(x).strip()

def build_prompt(row, include_answer=False):
    choices = row["choices"]
    choices_text = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    prompt = "<image>\n"
    prompt += "You are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint:    prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer:"

    if include_answer:
        ans = int(row["answer"])
        letter = CHOICE_LETTERS[ans]
        prompt += f" {letter}. {choices[ans]}"

    return prompt

def load_image(row):
    path = DATA_ROOT / row["image_path"]
    return Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)

# Test
row = train_df.iloc[0]
p = build_prompt(row, include_answer=True)
print(p[-100:])

 the male's tadpoles will become adult frogs

Answer: C. the male's tadpoles will become adult frogs


## Cell 5: Load Model + LoRA

In [12]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} (limit: 5,000,000)")
assert trainable <= 5_000_000

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


trainable params: 4,784,128 || all params: 512,266,432 || trainable%: 0.9339
Trainable: 4,784,128 (limit: 5,000,000)


In [13]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,}")

Trainable: 4,784,128


## Cell 6: Dataset & Collator

In [14]:
class VQATrainDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_image(row)
        prompt_text = build_prompt(row, include_answer=False)
        full_text = build_prompt(row, include_answer=True)

        # Process individually (avoids batched processor truncation issues)
        full_inputs = processor(text=[full_text], images=[image], return_tensors="pt", truncation=False)
        prompt_inputs = processor(text=[prompt_text], images=[image], return_tensors="pt", truncation=False)

        prompt_len = prompt_inputs["input_ids"].shape[1]

        return {
            "input_ids": full_inputs["input_ids"].squeeze(0),
            "attention_mask": full_inputs["attention_mask"].squeeze(0),
            "pixel_values": full_inputs["pixel_values"].squeeze(0),
            "prompt_len": prompt_len,
        }

def collate_train(batch):
    max_len = max(x["input_ids"].shape[0] for x in batch)
    pad_id = processor.tokenizer.pad_token_id

    input_ids_list, attention_mask_list, labels_list, pixel_values_list = [], [], [], []

    for x in batch:
        seq_len = x["input_ids"].shape[0]
        pad_len = max_len - seq_len
        prompt_len = x["prompt_len"]

        input_ids = torch.cat([
            torch.full((pad_len,), pad_id, dtype=x["input_ids"].dtype),
            x["input_ids"],
        ])
        attention_mask = torch.cat([
            torch.zeros(pad_len, dtype=x["attention_mask"].dtype),
            x["attention_mask"],
        ])

        # Mask everything except the answer tokens (after prompt)
        labels = torch.full_like(input_ids, -100)
        # Answer starts at position (pad_len + prompt_len) in padded sequence
        answer_start = pad_len + prompt_len
        labels[answer_start:] = input_ids[answer_start:]

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)
        pixel_values_list.append(x["pixel_values"])

    return {
        "input_ids": torch.stack(input_ids_list),
        "attention_mask": torch.stack(attention_mask_list),
        "labels": torch.stack(labels_list),
        "pixel_values": torch.stack(pixel_values_list),
    }

# Verify
print("Testing collator...")
_test_ds = VQATrainDataset(train_df.head(4))
_test_batch = collate_train([_test_ds[j] for j in range(4)])
for i in range(4):
    n_loss = (_test_batch["labels"][i] != -100).sum().item()
    total = (_test_batch["attention_mask"][i] == 1).sum().item()
    answer_tokens = _test_batch["labels"][i][_test_batch["labels"][i] != -100]
    decoded = processor.tokenizer.decode(answer_tokens)
    print(f"  Sample {i}: {n_loss} loss tokens out of {total}, answer='{decoded}'")

Testing collator...
  Sample 0: 11 loss tokens out of 1682, answer=' C. the male's tadpoles will become adult frogs'
  Sample 1: 9 loss tokens out of 1654, answer=' A. the female's offspring will live longer'
  Sample 2: 10 loss tokens out of 1654, answer=' B. the lioness's cubs will survive attacks'
  Sample 3: 8 loss tokens out of 1657, answer=' B. the gull's offspring will survive'


## Cell 7: Loss Trend Check

In [15]:
subset = train_df.sample(frac=0.10, random_state=SEED).reset_index(drop=True)
ds_check = VQATrainDataset(subset)
loader_check = DataLoader(ds_check, batch_size=BATCH_SIZE_TRAIN, shuffle=True,
                           collate_fn=collate_train, num_workers=0)

model.train()
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01
)
losses = []
optimizer.zero_grad()

N_CHECK_STEPS = 200
for step, batch in enumerate(loader_check):
    if step >= N_CHECK_STEPS: break
    batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
    out = model(**batch)
    loss = out.loss / GRAD_ACCUM_STEPS
    loss.backward()
    if (step + 1) % GRAD_ACCUM_STEPS == 0:
        optimizer.step(); optimizer.zero_grad(set_to_none=True)
    losses.append(out.loss.item())
    if (step+1) % 32 == 0:
        print(f"Step {step+1}: loss={losses[-1]:.4f} avg32={np.mean(losses[-32:]):.4f}")

w = 32
print(f"\nFirst {w} avg: {np.mean(losses[:w]):.4f}")
print(f"Last {w} avg:  {np.mean(losses[-w:]):.4f}")

model.eval(); model.config.use_cache = True
del optimizer, loader_check, ds_check
gc.collect(); torch.cuda.empty_cache()

Step 32: loss=0.0552 avg32=0.5026
Step 64: loss=0.4508 avg32=0.8831
Step 96: loss=0.9360 avg32=0.6641
Step 128: loss=0.0882 avg32=0.5041
Step 160: loss=0.9429 avg32=0.4318
Step 192: loss=0.0203 avg32=0.4126

First 32 avg: 0.5026
Last 32 avg:  0.4017


## Cell 8: Full Training with Checkpoints

In [16]:
# Reload fresh model
del model; gc.collect(); torch.cuda.empty_cache()

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} (limit: 5,000,000)")
assert trainable <= 5_000_000

model.train()
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

ds_full = VQATrainDataset(train_df)
loader_full = DataLoader(ds_full, batch_size=BATCH_SIZE_TRAIN, shuffle=True,
                          collate_fn=collate_train, num_workers=2, pin_memory=True)

total_steps = len(loader_full) * EPOCHS
opt_steps = total_steps // GRAD_ACCUM_STEPS

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01
)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=max(1, int(opt_steps * WARMUP_RATIO)),
    num_training_steps=opt_steps,
)

loss_history = []
best_avg_loss = float("inf")
optimizer.zero_grad()
global_step = 0

CHECKPOINT_EVERY = 500

for epoch in range(EPOCHS):
    pbar = tqdm(loader_full, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for step, batch in enumerate(pbar):
        batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss / GRAD_ACCUM_STEPS
        loss.backward()

        if (global_step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        loss_history.append(out.loss.item())
        global_step += 1

        if global_step % 100 == 0:
            pbar.set_postfix(loss=f"{loss_history[-1]:.4f}",
                             avg100=f"{np.mean(loss_history[-100:]):.4f}",
                             lr=f"{scheduler.get_last_lr()[0]:.2e}")

        if global_step % CHECKPOINT_EVERY == 0:
            current_avg = np.mean(loss_history[-CHECKPOINT_EVERY:])
            if current_avg < best_avg_loss:
                best_avg_loss = current_avg
                print(f"\n  💾 New best avg loss: {best_avg_loss:.4f} at step {global_step}")
                SAVE_DIR.mkdir(parents=True, exist_ok=True)
                model.save_pretrained(SAVE_DIR)
                processor.save_pretrained(SAVE_DIR)
                import shutil
                DRIVE_SAVE.mkdir(parents=True, exist_ok=True)
                if DRIVE_SAVE.exists(): shutil.rmtree(DRIVE_SAVE)
                shutil.copytree(SAVE_DIR, DRIVE_SAVE)
                print(f"  Saved to {DRIVE_SAVE}")

# Final save
model.eval(); model.config.use_cache = True
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
import shutil
if DRIVE_SAVE.exists(): shutil.rmtree(DRIVE_SAVE)
shutil.copytree(SAVE_DIR, DRIVE_SAVE)

torch.cuda.empty_cache()
print(f"\nDone! First 50 avg: {np.mean(loss_history[:50]):.4f}, Last 50 avg: {np.mean(loss_history[-50:]):.4f}")
print(f"Best checkpoint avg loss: {best_avg_loss:.4f}")

Trainable: 4,784,128 (limit: 5,000,000)


Epoch 1/3:   0%|          | 0/3109 [00:00<?, ?it/s]


  💾 New best avg loss: 0.5479 at step 500
  Saved to /content/drive/MyDrive/lora_v3_best

  💾 New best avg loss: 0.1943 at step 1000
  Saved to /content/drive/MyDrive/lora_v3_best

  💾 New best avg loss: 0.1272 at step 1500
  Saved to /content/drive/MyDrive/lora_v3_best

  💾 New best avg loss: 0.1027 at step 2500
  Saved to /content/drive/MyDrive/lora_v3_best


Epoch 2/3:   0%|          | 0/3109 [00:00<?, ?it/s]


  💾 New best avg loss: 0.0840 at step 3500
  Saved to /content/drive/MyDrive/lora_v3_best

  💾 New best avg loss: 0.0820 at step 6000
  Saved to /content/drive/MyDrive/lora_v3_best


Epoch 3/3:   0%|          | 0/3109 [00:00<?, ?it/s]


  💾 New best avg loss: 0.0711 at step 6500
  Saved to /content/drive/MyDrive/lora_v3_best

  💾 New best avg loss: 0.0692 at step 9000
  Saved to /content/drive/MyDrive/lora_v3_best

Done! First 50 avg: 0.6936, Last 50 avg: 0.0964
Best checkpoint avg loss: 0.0692


## Cell 9: Letter Log-Prob Inference

In [17]:
def get_candidate_token_ids(tokenizer):
    cids = {}
    for letter in CHOICE_LETTERS:
        forms = [letter, " "+letter, "\n"+letter, letter+".", " "+letter+"."]
        ids = set()
        for f in forms:
            enc = tokenizer.encode(f, add_special_tokens=False)
            if enc: ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids

candidate_ids = get_candidate_token_ids(processor.tokenizer)

def predict_batch(df_batch):
    images = [load_image(row) for _, row in df_batch.iterrows()]
    prompts = [build_prompt(row, include_answer=False) for _, row in df_batch.iterrows()]

    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]

    log_probs = torch.log_softmax(logits, dim=-1)
    preds, scores_all = [], []

    for i, (_, row) in enumerate(df_batch.iterrows()):
        scores = []
        for ci in range(len(row["choices"])):
            tids = candidate_ids[CHOICE_LETTERS[ci]]
            scores.append(log_probs[i, tids].max().item())
        preds.append(int(np.argmax(scores)))
        scores_all.append(scores)

    return preds, scores_all

## Cell 10: Validate

In [18]:
model.eval()
gc.collect(); torch.cuda.empty_cache()

N_VAL = 200
eval_df = val_df.copy()
eval_df = eval_df.sample(n=min(N_VAL, len(eval_df)), random_state=SEED).reset_index(drop=True)

all_preds, all_scores = [], []
for start in tqdm(range(0, len(eval_df), BATCH_SIZE_INFER), desc="Val"):
    batch = eval_df.iloc[start:start+BATCH_SIZE_INFER]
    p, s = predict_batch(batch)
    all_preds.extend(p); all_scores.extend(s)
    torch.cuda.empty_cache()

eval_df["pred"] = all_preds
eval_df["correct"] = eval_df["pred"] == eval_df["answer"]
acc = eval_df["correct"].mean()
print(f"\nValidation accuracy: {acc:.4f} ({eval_df['correct'].sum()}/{len(eval_df)})")
print(f"Target: > 0.678")

Val:   0%|          | 0/50 [00:00<?, ?it/s]


Validation accuracy: 0.6400 (128/200)
Target: > 0.678


## Cell 11: Generate submission.csv

In [19]:
all_preds = []
for start in tqdm(range(0, len(test_df), BATCH_SIZE_INFER), desc="Test"):
    batch = test_df.iloc[start:start+BATCH_SIZE_INFER]
    p, _ = predict_batch(batch)
    all_preds.extend(p)
    torch.cuda.empty_cache()

submission = pd.DataFrame({"id": test_df["id"], "answer": all_preds})
submission.to_csv("/content/submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/submission.csv", index=False)
print(f"Saved submission.csv ({len(submission)} rows)")
print(submission.head())

Test:   0%|          | 0/252 [00:00<?, ?it/s]

Saved submission.csv (1008 rows)
           id  answer
0  test_01750       1
1  test_00128       0
2  test_02891       0
3  test_02425       3
4  test_00930       1


In [20]:
from google.colab import files
files.download("/content/submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>